In [2]:
# Importing the global libraries
import importlib, sys, os, pandas as pd
# from dotenv import load_dotenv
import pyspark.sql.types as T
import pyspark.sql as sql
from pyspark.sql import SparkSession, functions as F
import numpy as np
import datetime as dt

spark = (
	SparkSession.builder
		.appName("new_start")
		.getOrCreate()
)

#Mandatory
importlib.reload(importlib)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
stations_df = spark.read.csv("sumo_data/stations.csv", header=True, sep=";", inferSchema=True)
station_to_station_df = spark.read.csv("sumo_data/station_to_station.csv", header=True, sep=";", inferSchema=True)
punctuality_data_df = spark.read.csv("sumo_data/punctuality/202501.csv", header=True, sep=";", inferSchema=True)

In [4]:
START_DATE = "2025-01-01"					# Simulation start date
START_TIME = "04:00:00"						# Simulation start time
NB_DAYS = 5									# Number of days to simulate
EDGES_MAX_SPEED = 27.78						# Max speed of the edges in m/s (100 km/h)
TRAIN_SPEED = 33.33							# Train speed in m/s (120 km/h)
START_DATETIME = dt.datetime.strptime(
	f"{START_DATE} {START_TIME}", "%Y-%m-%d %H:%M:%S"
)
END_DATETIME = START_DATETIME + dt.timedelta(days=NB_DAYS)
DIRECTORY = "new_start"

filenames = {
	"stations" : f"{DIRECTORY}/stations.nod.xml",
	"edges" : f"{DIRECTORY}/edges.edg.xml",
	"routes" : f"{DIRECTORY}/routes.rou.xml",
	"network" : f"{DIRECTORY}/network.net.xml",
	"platforms" : f"{DIRECTORY}/platforms.add.xml",
	"schedule" : f"{DIRECTORY}/schedule.trips.xml",
	"config" : f"{DIRECTORY}/config.sumocfg",
	"train" : f"{DIRECTORY}/trains.add.xml"
}

In [6]:
stations_df.printSchema()
stations_df.show(5)

root
 |-- ID: integer (nullable = true)
 |-- Geo_x: double (nullable = true)
 |-- Geo_y: double (nullable = true)
 |-- Symbolic_name: string (nullable = true)
 |-- Name_FR_short: string (nullable = true)
 |-- Name_FR_full: string (nullable = true)

+----+---------+--------+-------------+-------------+------------+
|  ID|    Geo_x|   Geo_y|Symbolic_name|Name_FR_short|Name_FR_full|
+----+---------+--------+-------------+-------------+------------+
| 841|50.932897|4.216581|          LHL|       Mollem|      Mollem|
| 246|50.527312|3.526278|          FCA|   Callenelle|  Callenelle|
| 457|50.726468|4.513842|          MGV|       Genval|      Genval|
| 782|50.614049|3.800495|          FFA|       Maffle|      Maffle|
|1102|50.528178|5.219617|          LHY|       Statte|      Statte|
+----+---------+--------+-------------+-------------+------------+
only showing top 5 rows


In [7]:
station_to_station_df.printSchema()
station_to_station_df.show(5)

root
 |-- Departure_station_id: integer (nullable = true)
 |-- Arrival_station_id: integer (nullable = true)
 |-- Geo_x: double (nullable = true)
 |-- Geo_y: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- Coordinates: string (nullable = true)

+--------------------+------------------+---------+--------+--------+--------------------+
|Departure_station_id|Arrival_station_id|    Geo_x|   Geo_y|Distance|         Coordinates|
+--------------------+------------------+---------+--------+--------+--------------------+
|                 841|               821|50.943277|4.221739|    2.46|[[50.932897, 4.21...|
|                 246|               958|50.520529|3.559228|    4.96|[[50.513613, 3.59...|
|                 457|               672|50.732192|4.505572|    1.74|[[50.73811, 4.497...|
|                 782|                77|50.621327|3.788608|    2.34|[[50.614049, 3.80...|
|                1102|               592|50.526327|5.226778|    1.18|[[50.527376, 5.23...|
+----

In [8]:
punctuality_data_df.printSchema()
punctuality_data_df.show(5)

root
 |-- TRAIN_NO: integer (nullable = true)
 |-- RELATION: string (nullable = true)
 |-- TRAIN_SERV: string (nullable = true)
 |-- STOPPING_PLACE_ID: integer (nullable = true)
 |-- THOP1_COD: string (nullable = true)
 |-- LINE_NO_DEP: string (nullable = true)
 |-- DELAY_ARR: integer (nullable = true)
 |-- DELAY_DEP: integer (nullable = true)
 |-- CIRC_TYP: integer (nullable = true)
 |-- RELATION_DIRECTION: string (nullable = true)
 |-- LINE_NO_ARR: string (nullable = true)
 |-- REAL_DATE_ARR: string (nullable = true)
 |-- REAL_DATE_DEP: string (nullable = true)
 |-- PLANNED_DATETIME_ARR: timestamp (nullable = true)
 |-- PLANNED_DATETIME_DEP: timestamp (nullable = true)

+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|THOP1_COD|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|CIRC_TYP|  RELATION_DIRECT

In [5]:
schema = T.ArrayType(T.ArrayType(T.DoubleType()))

station_to_station_df = station_to_station_df.withColumn(
	"Coordinates",
	F.from_json(F.col("Coordinates"), schema)
)

station_to_station_df.printSchema()

root
 |-- Departure_station_id: integer (nullable = true)
 |-- Arrival_station_id: integer (nullable = true)
 |-- Geo_x: double (nullable = true)
 |-- Geo_y: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- Coordinates: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: double (containsNull = true)



In [6]:
# Définition d'une fenêtre partitionnée par TRAIN_NO et ordonnée par PLANNED_DATETIME_DEP

window = sql.Window.partitionBy("TRAIN_NO","REAL_DATE_DEP") \
	.orderBy("PLANNED_DATETIME_DEP")

punctuality_data_df = (
	punctuality_data_df
	.withColumn(
		"NEXT_STOPPING_PLACE_ID",
		F.lead("STOPPING_PLACE_ID").over(window)
	)
)

# punctuality_data_df = (
# 	punctuality_data_df.alias("a")
# 	# .join(
# 	# 	stations_df.select("ID").alias("b"), 
# 	# 	F.col("a.NEXT_STOPPING_PLACE_ID") == F.col("b.ID"), 
# 	# 	"left")
# 	# .drop("ID")
# )
punctuality_data_df.show(20)

+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|THOP1_COD|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|CIRC_TYP|  RELATION_DIRECTION|LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|NEXT_STOPPING_PLACE_ID|
+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|      10|     ICE| SNCB/NMBS|              825|        D|         37|     1795|     1795|       1|ICE: FRANKFURT(MA...|         37|   03-01-2025|   03-01-2025| 2025-01-03 20:28:00| 2025-01-03 20:28:00|                   266|
|      10|     ICE| SNCB/NMBS|              266|        P|         37|     1875|     1875|      

In [7]:
stations_dict = {}
for row in station_to_station_df.collect():
	departure_station, arrival_station = row['Departure_station_id'], row['Arrival_station_id']
	if departure_station not in stations_dict:
		stations_dict[departure_station] = set()
	stations_dict[departure_station].add(arrival_station)
for key in stations_dict:
	print(f"{key} -> {stations_dict[key]}")

841 -> {74, 821}
246 -> {958, 807}
457 -> {672, 997}
782 -> {832, 77}
1102 -> {592, 118}
157 -> {184, 324, 1151}
1912 -> {1761, 380, 189}
143 -> {708, 1348}
901 -> {956, 421}
1761 -> {1912, 826, 1219, 380}
952 -> {989, 583}
578 -> {742, 1079}
754 -> {148, 438}
733 -> {835, 470}
619 -> {401, 971}
1146 -> {1176, 384}
493 -> {1230, 190}
220 -> {455, 415, 2089, 815, 725, 504, 217, 1021, 414, 223}
900 -> {384, 684}
494 -> {345, 1228}
427 -> {560, 1154}
107 -> {840, 708}
249 -> {480, 951}
560 -> {427, 868}
132 -> {9, 187, 686}
1185 -> {477, 1157}
347 -> {67, 910}
1278 -> {64, 819, 58, 1084, 30}
736 -> {404, 1149}
1060 -> {938, 1253}
1266 -> {74, 621}
1091 -> {1843, 1206}
1157 -> {992, 1185}
936 -> {739, 975, 855, 762, 252}
118 -> {25, 1102}
19 -> {1090, 523}
130 -> {449, 748}
27 -> {2011, 266, 208, 1202, 726, 1147}
818 -> {114, 788}
905 -> {1066, 188}
1176 -> {1146, 715}
764 -> {58, 37, 38, 1839}
8 -> {136, 797}
142 -> {732, 814}
728 -> {730, 726}
267 -> {266, 1159}
148 -> {754, 1031}
192 ->

In [8]:
test_stations = [
	288,	# Cour-sur-Heure
	147, 	# Berzée
]

# test_stations = [
# 	291,	# Couvin
# 	798, 	# Mariembourg
# 	961, 	# Philippeville
# 	2199
# ]

In [9]:
for station in test_stations :
	if station in stations_dict :
		if len(stations_dict[station]) <= 2 :
			print(f"The station {station} has {len(stations_dict[station])} neighbors: {stations_dict[station]}")
		else :
			print(f"The station {station} has more than 2 neighbors: {len(stations_dict[station])} neighbors")
	else :
		print(f"The station {station} is not in the dataset")

The station 288 has 2 neighbors: {514, 147}
The station 147 has 2 neighbors: {976, 288}


In [10]:
print(START_DATETIME.strftime("%Y-%m-%d %H:%M:%S"))
print(END_DATETIME.strftime("%Y-%m-%d %H:%M:%S"))

subset_punctuality_data_df = (punctuality_data_df
	.filter(
		(F.col("PLANNED_DATETIME_DEP") >= F.lit(START_DATETIME.strftime("%Y-%m-%d %H:%M:%S"))) & 
		(F.col("PLANNED_DATETIME_DEP") <= F.lit(END_DATETIME.strftime("%Y-%m-%d %H:%M:%S")))
	).orderBy(F.col("PLANNED_DATETIME_DEP").desc(), "TRAIN_NO", "REAL_DATE_DEP")
)
subset_punctuality_data_df.show(20)

2025-01-01 04:00:00
2025-01-06 04:00:00


+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|THOP1_COD|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|CIRC_TYP|  RELATION_DIRECTION|LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|NEXT_STOPPING_PLACE_ID|
+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|     522|   IC 01| SNCB/NMBS|               31|        =|         36|      -16|        6|       1|IC 01: OOSTENDE -...|         36|   06-01-2025|   06-01-2025| 2025-01-06 01:36:00| 2025-01-06 01:37:00|                   929|
|     522|   IC 01| SNCB/NMBS|              155|        D|         36|       28|       28|      

In [11]:
additional_stations = []
for station in test_stations :
	for s in stations_dict[station] :
		if s not in test_stations and s not in additional_stations :
			additional_stations.append(s)
print(additional_stations)

[514, 976]


In [12]:
subset_punctuality_data_df = (subset_punctuality_data_df
.filter(
		F.col("STOPPING_PLACE_ID").isin(test_stations) |
		F.col("NEXT_STOPPING_PLACE_ID").isin(test_stations)
	).sort("REAL_DATE_DEP", "TRAIN_NO", "PLANNED_DATETIME_DEP", ascending=True)
)
subset_punctuality_data_df.show(20)

+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|THOP1_COD|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|CIRC_TYP|  RELATION_DIRECTION|LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|NEXT_STOPPING_PLACE_ID|
+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|    3908|    L C4| SNCB/NMBS|              514|        =|        132|      135|      120|       1|L C4: CHARLEROI-C...|        132|   01-01-2025|   01-01-2025| 2025-01-01 08:30:00| 2025-01-01 08:31:00|                   288|
|    3908|    L C4| SNCB/NMBS|              288|        =|        132|      105|       45|      

In [122]:
print(subset_punctuality_data_df.count())

502


In [13]:
hashmap_stations = {
	1155 : [404, 736, 1149, 1189],
	516 : [190, 493, 507, 1230],
	264 : [16, 700],
	650 : [868],
	1419 : [376, 539, 632, 637, 1084],
	524 : [139, 523, 1061],
	269 : [789, 894, 1062],
	2061 : [1048],
	528 : [70, 489, 723, 752, 968],
	913 : [313, 610, 611, 767, 789, 895, 995, 1009],
	533 : [77, 427, 719, 1048, 1154, 1260],
	2199 : [798, 961, 1254],
	921 : [535, 908, 920, 1136],
	794 : [413, 747, 791],
	283 : [604, 606, 649, 1005],
	925 : [31, 267, 421, 901, 1159],
	1055 : [191, 218, 227, 383, 812, 826],
	800 : [363, 1048, 1192, 1224],
	1057 : [221, 363, 1048, 1192, 1979],
	674 : [201, 391, 471, 673, 784, 790, 1048, 1744],
	162 : [139, 151, 1088],
	33 : [368, 648, 916, 1174, 1260],
	164 : [347, 458, 600, 910, 923],
	675 : [217, 376, 764, 1839],
	551 : [205, 342, 550, 1092, 1160],
	1577 : [424, 515, 620, 809, 870],
	42 : [139, 546, 554, 977],
	937 : [153, 252, 739, 855, 936],
	428 : [1067, 2011],
	557 : [22, 361, 402, 530],
	2222 : [436, 554, 924, 1232],
	308 : [82, 515, 620, 1125],
	949 : [210, 931, 1261],
	950 : [210, 1195],
	698 : [560, 649, 868],
	1211 : [455, 682],
	1724 : [139, 320, 335, 447, 449, 455],
	1982 : [75, 272, 287, 896, 1043],
	1089 : [138, 151, 906, 1088],
	456 : [130, 447, 449, 1248],
	76 : [61, 911],
	333 : [126, 435, 480, 589],
	2126 : [37, 203, 715, 738, 911, 1218],
	718 : [9, 507, 553, 715, 1238],
	1110 : [84, 992, 996, 1157],
	731 : [27, 562, 726, 728, 1063, 2011],
	93 : [10, 220, 317, 320, 455, 1212],
	734 : [77, 139, 221, 458, 562, 726, 730, 733, 835, 895, 1125],
	1246 : [84, 277, 375, 726, 974, 996],
	225 : [1768, 2089],
	614 : [25, 118, 262, 1102],
	358 : [726, 728, 1067, 2011],
	745 : [412, 664, 744, 1139],
	233 : [157, 324, 523, 1151],
	2027 : [258, 259, 791],
	618 : [255, 620, 809, 870],
	749 : [130, 138, 748, 1073, 1128, 1265],
	108 : [436, 750, 840, 1666],
	1263 : [212, 743],
	236 : [148, 220, 895, 995, 1031],
	109 : [941],
	1011 : [400, 406, 895, 1009],
	117 : [37, 139, 324, 326, 523, 636, 1061, 1141, 1238],
	885 : [205, 805, 902, 1092],
	1273 : [565, 1231, 1348, 1350],
	506 : [128, 504, 759],
	1148 : [619, 726, 971, 1063],
	638 : [27, 210, 266, 399, 726, 929, 1202],
	1023 : [220, 232, 415, 759, 1021]
}

In [14]:
trips = []
trip_number = None
trip = []
for row in subset_punctuality_data_df.collect() :
	if trip_number is None :
		# print(f"First train : {row['TRAIN_NO']}")
		trip_number = row["TRAIN_NO"]
	
	delta = (row["PLANNED_DATETIME_DEP"] - START_DATETIME).total_seconds()
	info = {
		"departure_station" : row["STOPPING_PLACE_ID"],
		"arrival_station" : row["NEXT_STOPPING_PLACE_ID"],
		"sumo_time" : int(delta)
	}

	in_hashmap_s1, in_hashmap_s2 = (info["departure_station"] in hashmap_stations, 
		info["arrival_station"] in hashmap_stations)

	if in_hashmap_s1 and in_hashmap_s2 :
		found = False
		for s1 in hashmap_stations[info["departure_station"]] :
			for s2 in hashmap_stations[info["arrival_station"]] :
				if s2 in stations_dict[s1] :
					info["departure_station"] = s1
					info["arrival_station"] = s2
				break
			if found : break
	elif in_hashmap_s1 and not in_hashmap_s2 :
		for s1 in hashmap_stations[info["departure_station"]] :
			if info["arrival_station"] in stations_dict[s1] :
				info["departure_station"] = s1
				break
	elif not in_hashmap_s1 and in_hashmap_s2 :
		for s2 in hashmap_stations[info["arrival_station"]] :
			if s2 in stations_dict[info["departure_station"]] :
				info["arrival_station"] = s2
				break

	if trip_number == row["TRAIN_NO"] :
		# print(f"Adding info : {info}")
		trip.append(info)
	else :
		# print(f"trip number from {trip_number} to {row['TRAIN_NO']}")
		# print(f"Current trip has {len(trip)} stops")
		# print(f"test_stations: {test_stations}")
		if len(trip) >= len(test_stations) :
			# print(f"Adding trip for train {trip_number} with {len(trip)} stops")
			trip.sort(key=lambda x: x["sumo_time"])
			trips.append(trip)
			trip = [info]
			trip_number = row["TRAIN_NO"]

if trip and len(trip) >= len(test_stations) :
	trip.sort(key=lambda x: x["sumo_time"])
	trips.append(trip)
trips.sort(key=lambda x: x[0]["sumo_time"])

len_trips : int = len(trips)
for i in range(len(trips)) :
	trip : list[dict] = trips[i]
	j : int = 0
	while j < len(trip) :
		if trip[j]["departure_station"] is None or trip[j]["arrival_station"] is None :
			trip.pop(j)
		else :
			j += 1
	t : int = 0
	while t < len(trip) - 1 :
		t1 = trip[t]
		t2 = trip[t + 1]
		if t1["departure_station"] == t2["departure_station"] or t1["arrival_station"] == t2["arrival_station"] :
			trip[t]["sumo_time"] = (t1["sumo_time"] + t2["sumo_time"]) / 2
			trip.pop(t + 1)
		else : 
			t += 1
trips.sort(key=lambda x: x[0]["sumo_time"])

print(f"Number of trips: {len(trips)}")
for trip in trips :
	print(trip)

Number of trips: 128
[{'departure_station': 976, 'arrival_station': 147, 'sumo_time': 12060}, {'departure_station': 147, 'arrival_station': 288, 'sumo_time': 12240}, {'departure_station': 288, 'arrival_station': 514, 'sumo_time': 12480}]
[{'departure_station': 514, 'arrival_station': 288, 'sumo_time': 16260}, {'departure_station': 288, 'arrival_station': 147, 'sumo_time': 16500}, {'departure_station': 147, 'arrival_station': 976, 'sumo_time': 16740}]
[{'departure_station': 976, 'arrival_station': 147, 'sumo_time': 19260}, {'departure_station': 147, 'arrival_station': 288, 'sumo_time': 19440}, {'departure_station': 288, 'arrival_station': 514, 'sumo_time': 19680}]
[{'departure_station': 514, 'arrival_station': 288, 'sumo_time': 23460}, {'departure_station': 288, 'arrival_station': 147, 'sumo_time': 23700}, {'departure_station': 147, 'arrival_station': 976, 'sumo_time': 23940}]
[{'departure_station': 976, 'arrival_station': 147, 'sumo_time': 26460}, {'departure_station': 147, 'arrival_st

In [15]:
for station in test_stations :
	print(f"Station {station} has {sum(1 for trip in trips if any(info['departure_station'] == station or info['arrival_station'] == station for info in trip))} trips passing through it")

Station 288 has 128 trips passing through it
Station 147 has 128 trips passing through it


In [16]:
s_start = test_stations[0]
s_end = test_stations[-1]

sim_stations = [s_start, s_end]

sim_stations.extend(stations_dict[s_start])
sim_stations.extend(stations_dict[s_end])
sim_stations = list(set(sim_stations))

subset_stations_df = stations_df.filter(F.col("ID").isin(sim_stations))
subset_stations_df.show()

+---+---------+--------+-------------+-------------+--------------+
| ID|    Geo_x|   Geo_y|Symbolic_name|Name_FR_short|  Name_FR_full|
+---+---------+--------+-------------+-------------+--------------+
|147|50.285623|4.405998|          LBZ|       Berzée|        Berzée|
|288|50.300903|4.392088|          LUH|   Cour-Heure|Cour-sur-Heure|
|514|50.320542|4.405047|          LHH|  Ham-s-Heure| Ham-sur-Heure|
|976|50.268758|4.430333|          PRY|          Pry|           Pry|
+---+---------+--------+-------------+-------------+--------------+



In [17]:
subset_station_to_station_df = station_to_station_df.filter(
	(F.col("Departure_station_id").isin(sim_stations)) &
	(F.col("Arrival_station_id").isin(sim_stations))
)
subset_station_to_station_df.show()

+--------------------+------------------+---------+--------+--------+--------------------+
|Departure_station_id|Arrival_station_id|    Geo_x|   Geo_y|Distance|         Coordinates|
+--------------------+------------------+---------+--------+--------+--------------------+
|                 147|               976|50.276045|4.417382|    2.61|[[50.285623, 4.40...|
|                 288|               514|50.310754|4.398718|     2.4|[[50.320542, 4.40...|
|                 147|               288| 50.29186|4.395707|    2.32|[[50.300903, 4.39...|
|                 514|               288|50.310754|4.398718|     2.4|[[50.320542, 4.40...|
|                 976|               147|50.276045|4.417382|    2.61|[[50.285623, 4.40...|
|                 288|               147| 50.29186|4.395707|    2.32|[[50.300903, 4.39...|
+--------------------+------------------+---------+--------+--------+--------------------+



In [18]:
stations_str = (
	'<?xml version="1.0" encoding="UTF-8"?>\n' + 
	'<nodes>\n' +
	"".join(f'\t<node id="{info["ID"]}" x="{info["Geo_x"]}" y="{info["Geo_y"]}" type="priority"/>\n' for info in [row 
		for row in subset_stations_df.collect()]) +
	'</nodes>'
)
print(stations_str)

with open(filenames["stations"], 'w', encoding = "utf-8") as f :
	f.write(stations_str)

<?xml version="1.0" encoding="UTF-8"?>
<nodes>
	<node id="147" x="50.285623" y="4.405998" type="priority"/>
	<node id="288" x="50.300903" y="4.392088" type="priority"/>
	<node id="514" x="50.320542" y="4.405047" type="priority"/>
	<node id="976" x="50.268758" y="4.430333" type="priority"/>
</nodes>


In [ ]:
edges_str = (
	'<?xml version="1.0" encoding="UTF-8"?>\n'
	'<edges>\n'
)

for info in [row for row in subset_station_to_station_df.collect()] :
	edges_str +=f'\t<edge id="{info["Departure_station_id"]}_{info["Arrival_station_id"]}" from="{info["Departure_station_id"]}" to="{info["Arrival_station_id"]}" priority="2" numLanes="1" length="{info["Distance"] * 1000}" speed="{EDGES_MAX_SPEED}" allow="rail" shape="'
	for coords in info["Coordinates"] :
		edges_str += f'{coords[0]},{coords[1]} '
	edges_str += f'"/>\n'
edges_str += '</edges>'
print(edges_str)

with open(filenames["edges"], 'w', encoding = "utf-8") as f :
	f.write(edges_str)

<?xml version="1.0" encoding="UTF-8"?>
<edges>
	<edge id="147_976" from="147" to="976" priority="2" numLanes="1" length="2610.0" speed="27.78" allow="rail" shape="50.285623,4.405998 50.285623,4.405998 50.285328,4.406411 50.28518,4.406616 50.285034,4.406816 50.284888,4.407012 50.284741,4.407204 50.284495,4.40751 50.28426,4.407786 50.284016,4.40805 50.283751,4.408319 50.283614,4.408453 50.283482,4.408578 50.283197,4.408837 50.282913,4.409083 50.281989,4.409826 50.27926,4.411971 50.278853,4.412327 50.278546,4.412658 50.278206,4.413095 50.273547,4.419439 50.273156,4.420004 50.272813,4.420546 50.272531,4.421038 50.272253,4.42157 50.271958,4.422198 50.271716,4.422767 50.270015,4.427108 50.268758,4.430333 "/>
	<edge id="288_514" from="288" to="514" priority="2" numLanes="1" length="2400.0" speed="27.78" allow="rail" shape="50.320542,4.405047 50.319606,4.404834 50.318698,4.404621 50.318406,4.404533 50.318278,4.404487 50.318153,4.404437 50.318017,4.404378 50.317883,4.404316 50.317723,4.404235 5

In [30]:
platforms_str = (
	'<?xml version="1.0" encoding="UTF-8"?>\n' + 
	'<additional>\n' 
	+"".join(f'\t<trainStop id="{info["Departure_station_id"]}->{info["Arrival_station_id"]}" lane="{info["Departure_station_id"]}_{info["Arrival_station_id"]}_0" name="{info["Departure_station_id"]}->{info["Arrival_station_id"]} platform" startPos="80" endPos="300" />\n'
		for info in [row for row in subset_station_to_station_df.collect()])
	+ '</additional>'
)
print(platforms_str)

with open(filenames["platforms"], 'w', encoding = "utf-8") as f :
	f.write(platforms_str)

<?xml version="1.0" encoding="UTF-8"?>
<additional>
	<trainStop id="147->976" lane="147_976_0" name="147->976 platform" startPos="80" endPos="300" />
	<trainStop id="288->514" lane="288_514_0" name="288->514 platform" startPos="80" endPos="300" />
	<trainStop id="147->288" lane="147_288_0" name="147->288 platform" startPos="80" endPos="300" />
	<trainStop id="514->288" lane="514_288_0" name="514->288 platform" startPos="80" endPos="300" />
	<trainStop id="976->147" lane="976_147_0" name="976->147 platform" startPos="80" endPos="300" />
	<trainStop id="288->147" lane="288_147_0" name="288->147 platform" startPos="80" endPos="300" />
</additional>


In [21]:
train_str =('<?xml version="1.0" encoding="UTF-8"?>\n' +
	'<additional>\n' + 
	f'\t<vType id="myTrain" vClass="rail" length="80" accel="1.0" decel="1.3" maxSpeed="{TRAIN_SPEED}" guiShape="rail"/>\n' +
	'</additional>'
)
print(train_str)

with open(filenames["train"], 'w', encoding = "utf-8") as f :
	f.write(train_str)

<?xml version="1.0" encoding="UTF-8"?>
<additional>
	<vType id="myTrain" vClass="rail" length="80" accel="1.0" decel="1.3" maxSpeed="33.33" guiShape="rail"/>
</additional>


In [23]:
trips_str = (
	'<?xml version="1.0" encoding="UTF-8"?>\n' + 
	'<routes>\n'
)
trip_number = 0
for trip in trips :
	# if trip[0]["departure_station"] in test_stations :
	depart_edge = f'{trip[0]["departure_station"]}_{trip[0]["arrival_station"]}'
	depart_time = trip[0]["sumo_time"]
	# else :
	# 	depart_edge = f'{trip[1]["departure_station"]}_{trip[1]["arrival_station"]}'
	# 	depart_time = trip[1]["sumo_time"]
	arrival_edge = f'{trip[-1]["departure_station"]}_{trip[-1]["arrival_station"]}'
	trips_str +=f'\t<trip id="trip_{trip_number}" depart="{depart_time}" from="{depart_edge}" to="{arrival_edge}" type="myTrain" />\n'
	trip_number += 1
trips_str += '</routes>'
print(trips_str)

with open(filenames["schedule"], 'w', encoding = "utf-8") as f :
	f.write(trips_str)

<?xml version="1.0" encoding="UTF-8"?>
<routes>
	<trip id="trip_0" depart="12060" from="976_147" to="288_514" type="myTrain" />
	<trip id="trip_1" depart="16260" from="514_288" to="147_976" type="myTrain" />
	<trip id="trip_2" depart="19260" from="976_147" to="288_514" type="myTrain" />
	<trip id="trip_3" depart="23460" from="514_288" to="147_976" type="myTrain" />
	<trip id="trip_4" depart="26460" from="976_147" to="288_514" type="myTrain" />
	<trip id="trip_5" depart="30660" from="514_288" to="147_976" type="myTrain" />
	<trip id="trip_6" depart="33660" from="976_147" to="288_514" type="myTrain" />
	<trip id="trip_7" depart="37860" from="514_288" to="147_976" type="myTrain" />
	<trip id="trip_8" depart="40860" from="976_147" to="288_514" type="myTrain" />
	<trip id="trip_9" depart="45060" from="514_288" to="147_976" type="myTrain" />
	<trip id="trip_10" depart="48060" from="976_147" to="288_514" type="myTrain" />
	<trip id="trip_11" depart="52260" from="514_288" to="147_976" type="my

In [24]:
network_command = " ".join([
	"netconvert",					
	"--node-files", f'{filenames["stations"]}',	
	"--edge-files", f'{filenames["edges"]}',	
	"--railway.signal.guess.by-stops", "true",			
	"--output-file", f'{filenames["network"]}',		
	"--proj.utm", "true"		
])
print(network_command)

netconvert --node-files new_start/stations.nod.xml --edge-files new_start/edges.edg.xml --railway.signal.guess.by-stops true --output-file new_start/network.net.xml --proj.utm true


In [31]:
routes_str = (
	'<?xml version="1.0" encoding="UTF-8"?>\n' + 
	'<routes>\n' +
	f'\t<vType id="myTrain" length="80.00" maxSpeed="{TRAIN_SPEED}" vClass="rail" guiShape="rail" accel="0.5" decel="1.0"/>\n' 
)
trip_cmp = 0

for trip in trips :
	# train_number = trip[0]["train_number"]
	# if trip[0]["departure_station"] not in test_stations :
	# 	trip = trip[1:]
	edges = [f"{info['departure_station']}_{info['arrival_station']}" for info in trip]
	routes_str += (
		f'\t<vehicle id="trai_{trip_cmp}" type="myTrain" depart="{float(trip[0]["sumo_time"])}">\n' +
		f'\t\t<route edges="{" ".join(edges)}" />\n'
	)
	trip_cmp += 1

	for info in trip[1:] :
		routes_str += (
			f'\t\t<stop trainStop="{info["departure_station"]}->{info["arrival_station"]}" until="{float(info["sumo_time"])}" duration="60"/>\n'
		)
	routes_str += '\t</vehicle>\n'
routes_str += '</routes>'
print(routes_str)

with open(filenames["routes"], 'w', encoding = "utf-8") as f :
	f.write(routes_str)

<?xml version="1.0" encoding="UTF-8"?>
<routes>
	<vType id="myTrain" length="80.00" maxSpeed="33.33" vClass="rail" guiShape="rail" accel="0.5" decel="1.0"/>
	<vehicle id="trai_0" type="myTrain" depart="12060.0">
		<route edges="976_147 147_288 288_514" />
		<stop trainStop="147->288" until="12240.0" duration="60"/>
		<stop trainStop="288->514" until="12480.0" duration="60"/>
	</vehicle>
	<vehicle id="trai_1" type="myTrain" depart="16260.0">
		<route edges="514_288 288_147 147_976" />
		<stop trainStop="288->147" until="16500.0" duration="60"/>
		<stop trainStop="147->976" until="16740.0" duration="60"/>
	</vehicle>
	<vehicle id="trai_2" type="myTrain" depart="19260.0">
		<route edges="976_147 147_288 288_514" />
		<stop trainStop="147->288" until="19440.0" duration="60"/>
		<stop trainStop="288->514" until="19680.0" duration="60"/>
	</vehicle>
	<vehicle id="trai_3" type="myTrain" depart="23460.0">
		<route edges="514_288 288_147 147_976" />
		<stop trainStop="288->147" until="23700.0" 

In [25]:
sumo_config_str = f"""<?xml version="1.0" encoding="UTF-8"?>
<configuration>
	<input>
		<net-file value="network.net.xml"/>
		<route-files value="routes.rou.xml"/>
		<additional-files value="platforms.add.xml"/>
	</input>
	<time>
		<begin value="{0}"/>
		<end value="{NB_DAYS * 24 * 3600}"/>
	</time>
	<report>
		<no-step-log value="true"/>
	</report>
	<output>
		<tripinfo-output value="tripinfo.xml"/>
		<stop-output value="stopinfo.xml"/>
		<summary-output value="summary.xml"/>
	</output>
</configuration>"""
print(sumo_config_str)

with open(filenames["config"], 'w', encoding = "utf-8") as f :
	f.write(sumo_config_str)

<?xml version="1.0" encoding="UTF-8"?>
<configuration>
	<input>
		<net-file value="network.net.xml"/>
		<route-files value="routes.rou.xml"/>
		<additional-files value="platforms.add.xml"/>
	</input>
	<time>
		<begin value="0"/>
		<end value="432000"/>
	</time>
	<report>
		<no-step-log value="true"/>
	</report>
	<output>
		<tripinfo-output value="tripinfo.xml"/>
		<stop-output value="stopinfo.xml"/>
		<summary-output value="summary.xml"/>
	</output>
</configuration>


In [27]:
sumo_command = f"sumo -c {filenames['config']}\nsumo-gui -c {filenames['config']}"
print(sumo_command)

sumo -c new_start/config.sumocfg
sumo-gui -c new_start/config.sumocfg
